# Data Cleaning and Covariance Estimation

This notebook builds clean return data and a robust covariance matrix for `INSTRUMENT_1` ... `INSTRUMENT_10` using:
- liquidity-aware filtering from `volumes.csv`
- robust winsorization of extreme returns
- EWMA covariance (recent data weighted higher)
- data-driven shrinkage to a diagonal target
- PSD projection (eigenvalue clipping)

All derived CSVs are exported to `dataCorr/outputs/`.


In [3]:
from pathlib import Path
import csv
import math
import numpy as np

DATA_DIR = Path('../data/2024-12-31')
OUT_DIR = Path('./outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.set_printoptions(suppress=True, precision=8)


In [4]:
def read_timeseries_csv(path):
    """Read CSV with first column=date and remaining numeric columns."""
    with path.open('r', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        dates = []
        rows = []
        for row in reader:
            dates.append(row[0])
            vals = []
            for x in row[1:]:
                x = x.strip()
                vals.append(float(x) if x != '' else np.nan)
            rows.append(vals)
    cols = [c.strip() for c in header[1:]]
    data = np.array(rows, dtype=float)
    return dates, cols, data


def write_timeseries_csv(path, dates, cols, data):
    with path.open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['date', *cols])
        for d, row in zip(dates, data):
            writer.writerow([d, *row.tolist()])


def write_matrix_csv(path, row_labels, col_labels, matrix):
    with path.open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['', *col_labels])
        for r, row in zip(row_labels, matrix):
            writer.writerow([r, *row.tolist()])


def robust_winsorize(x, k=8.0):
    """Column-wise MAD winsorization for return matrix x (NaNs allowed)."""
    y = x.copy()
    for j in range(y.shape[1]):
        col = y[:, j]
        med = np.nanmedian(col)
        mad = np.nanmedian(np.abs(col - med))
        sigma = 1.4826 * mad
        if not np.isfinite(sigma) or sigma <= 0:
            continue
        lo, hi = med - k * sigma, med + k * sigma
        y[:, j] = np.clip(col, lo, hi)
    return y


def ewma_covariance(x, halflife=60):
    """EWMA covariance on complete (no-NaN) return matrix x."""
    t, _ = x.shape
    lam = math.exp(math.log(0.5) / float(halflife))
    w = lam ** np.arange(t - 1, -1, -1, dtype=float)
    w = w / w.sum()
    mu = (w[:, None] * x).sum(axis=0)
    xc = x - mu
    cov = (xc * w[:, None]).T @ xc
    return 0.5 * (cov + cov.T)


def oas_shrinkage_intensity(x):
    """OAS-style shrinkage intensity computed from complete return matrix x."""
    t, p = x.shape
    if t <= 1:
        return 1.0
    s = np.cov(x, rowvar=False, ddof=1)
    tr_s = np.trace(s)
    tr_s2 = np.trace(s @ s)
    num = (1.0 - 2.0 / p) * tr_s2 + tr_s * tr_s
    den = (t + 1.0 - 2.0 / p) * (tr_s2 - (tr_s * tr_s) / p)
    if den <= 0:
        return 1.0
    alpha = num / den
    return float(np.clip(alpha, 0.0, 1.0))


def nearest_psd(mat, eps=1e-10):
    m = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(m)
    vals = np.clip(vals, eps, None)
    psd = vecs @ np.diag(vals) @ vecs.T
    return 0.5 * (psd + psd.T)


def covariance_to_correlation(cov):
    d = np.sqrt(np.clip(np.diag(cov), 1e-20, None))
    denom = np.outer(d, d)
    corr = cov / denom
    return np.clip(corr, -1.0, 1.0)


In [5]:
price_dates, instruments, prices = read_timeseries_csv(DATA_DIR / 'prices.csv')
vol_dates, vol_cols, volumes = read_timeseries_csv(DATA_DIR / 'volumes.csv')

assert price_dates == vol_dates, 'prices and volumes dates do not match'
expected_vol_cols = [f'{c}_vol' for c in instruments]
assert vol_cols == expected_vol_cols, 'volume columns are not aligned with prices'

assert np.all(prices > 0), 'prices must be strictly positive for log-returns'

print('instruments:', len(instruments))
print('rows:', len(price_dates), 'date range:', price_dates[0], 'to', price_dates[-1])


instruments: 10
rows: 2851 date range: 2017-01-03 to 2024-12-31


In [6]:
# 1) Liquidity-aware valid-return mask: return at date t is valid only if vol(t) and vol(t-1) are positive.
vol_positive = volumes > 0
vol_prev = np.vstack([np.zeros((1, volumes.shape[1]), dtype=bool), vol_positive[:-1]])
valid_return_mask = vol_positive & vol_prev

# 2) Build log returns aligned to date t (between t-1 and t).
log_prices = np.log(prices)
log_returns = log_prices[1:] - log_prices[:-1]
ret_dates = price_dates[1:]
valid_return_mask = valid_return_mask[1:]

# 3) Mask out returns where instrument was not liquid across the return interval.
ret_masked = log_returns.copy()
ret_masked[~valid_return_mask] = np.nan

# 4) Robust winsorization to suppress occasional spikes.
ret_clean = robust_winsorize(ret_masked, k=8.0)

# 5) Build common-liquid panel (all 10 instruments valid) for a full-rank covariance matrix.
common_liquid_rows = np.all(np.isfinite(ret_clean), axis=1)
ret_common = ret_clean[common_liquid_rows]
ret_common_dates = [d for d, keep in zip(ret_dates, common_liquid_rows) if keep]

if ret_common.shape[0] < 100:
    raise ValueError('Too few common-liquid return rows for stable covariance estimation.')

print('raw return rows:', log_returns.shape[0])
print('common-liquid rows used:', ret_common.shape[0])


raw return rows: 2850
common-liquid rows used: 1450


In [7]:
# Base estimators
cov_sample = np.cov(ret_common, rowvar=False, ddof=1)
cov_ewma = ewma_covariance(ret_common, halflife=60)

# Data-driven shrinkage intensity (OAS) applied to EWMA toward diagonal target
alpha = oas_shrinkage_intensity(ret_common)
diag_target = np.diag(np.diag(cov_ewma))
cov_shrunk = (1.0 - alpha) * cov_ewma + alpha * diag_target

# Ensure positive semidefinite and create annualized version
cov_final_daily = nearest_psd(cov_shrunk, eps=1e-10)
cov_final_annualized = cov_final_daily * 252.0
corr_final = covariance_to_correlation(cov_final_daily)

print(f'shrinkage alpha: {alpha:.4f}')
print('min eigenvalue (daily final):', np.linalg.eigvalsh(cov_final_daily).min())


shrinkage alpha: 0.0024
min eigenvalue (daily final): 2.273233606401575e-06


In [8]:
# Export cleaned panel and covariance outputs
write_timeseries_csv(OUT_DIR / 'returns_log_masked.csv', ret_dates, instruments, ret_masked)
write_timeseries_csv(OUT_DIR / 'returns_log_clean.csv', ret_dates, instruments, ret_clean)
write_timeseries_csv(OUT_DIR / 'returns_log_common_liquid.csv', ret_common_dates, instruments, ret_common)

write_matrix_csv(OUT_DIR / 'cov_sample_daily.csv', instruments, instruments, cov_sample)
write_matrix_csv(OUT_DIR / 'cov_ewma_daily.csv', instruments, instruments, cov_ewma)
write_matrix_csv(OUT_DIR / 'cov_final_daily.csv', instruments, instruments, cov_final_daily)
write_matrix_csv(OUT_DIR / 'cov_final_annualized.csv', instruments, instruments, cov_final_annualized)
write_matrix_csv(OUT_DIR / 'corr_final.csv', instruments, instruments, corr_final)

with (OUT_DIR / 'covariance_metadata.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['metric', 'value'])
    writer.writerow(['sample_start', ret_common_dates[0]])
    writer.writerow(['sample_end', ret_common_dates[-1]])
    writer.writerow(['rows_used', ret_common.shape[0]])
    writer.writerow(['instruments', ret_common.shape[1]])
    writer.writerow(['ewma_halflife_days', 60])
    writer.writerow(['shrinkage_alpha', alpha])

print('Wrote outputs to:', OUT_DIR.resolve())


Wrote outputs to: /Users/devrajkatkoria/Documents/man-imperial-algothon-2026/dataCorr/outputs


In [9]:
cov_final_daily


array([[ 0.00007314,  0.00009792,  0.00004846,  0.0000491 ,  0.00000629,
         0.00000488,  0.00002494,  0.00001173,  0.0001097 ,  0.0001493 ],
       [ 0.00009792,  0.0001467 ,  0.00006455,  0.00006842,  0.00000555,
         0.00000504,  0.00003222,  0.00001659,  0.00013836,  0.00019253],
       [ 0.00004846,  0.00006455,  0.00007873,  0.00007266,  0.00001826,
         0.00001079,  0.0000399 ,  0.0000264 ,  0.00007371,  0.00011012],
       [ 0.0000491 ,  0.00006842,  0.00007266,  0.00010822,  0.00001609,
         0.00001025,  0.00004823,  0.00004032,  0.00007606,  0.00009862],
       [ 0.00000629,  0.00000555,  0.00001826,  0.00001609,  0.00008402,
         0.00003651,  0.0000323 , -0.00002507, -0.00000854,  0.00000314],
       [ 0.00000488,  0.00000504,  0.00001079,  0.00001025,  0.00003651,
         0.00001881,  0.00001766, -0.00001097,  0.00000471,  0.00001301],
       [ 0.00002494,  0.00003222,  0.0000399 ,  0.00004823,  0.0000323 ,
         0.00001766,  0.00008977,  0.0000333 